# A Basic RAG System (framework-free, ChromaDB)
### Module 4 — RAG Foundations (part 2 of 2)

**Goal:** assemble a full RAG pipeline the way Omar Santos teaches it — ingest security documents into a **ChromaDB** vector store, **retrieve** by meaning, **augment** the prompt with the hits, and **generate** a grounded answer. Built with plain `chromadb` + `sentence-transformers` — **no LangChain** (that's the next module).

> **Sources:** Omar Santos — LinkedIn course *RAG, AI Apps, and AI Agents for Cybersecurity and Networking*; book Ch. 2; repo `part4_rag_examples` (`basic_rag_part1.py`, `basic_rag_part2.py`). Knowledge base: `ssrf.txt` + `cybersecurity_kb.md`.

> **Reproducibility:** ingestion + retrieval run **offline**. Generation uses OpenAI if `OPENAI_API_KEY` is set; otherwise the notebook prints the assembled prompt so every step still runs. (Do part 1, `embeddings_and_vector_db.ipynb`, first.)

## 🛠️ Setup

In [1]:
# If needed:  !pip install -q sentence-transformers chromadb openai
import os
from sentence_transformers import SentenceTransformer
import chromadb

embedder = SentenceTransformer("all-MiniLM-L6-v2")       # local, offline embedding model
def embed(texts):
    return embedder.encode(texts, normalize_embeddings=True).tolist()

print("Setup ready. Embedding model: all-MiniLM-L6-v2 (384-dim, local).")

C:\Users\huber\Documents\CyberAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2323.88it/s]

Setup ready. Embedding model: all-MiniLM-L6-v2 (384-dim, local).


# 📄 Step 1 — Load & Chunk the Knowledge Base

- Load security documents (`ssrf.txt` + `cybersecurity_kb.md`)
- Split each into overlapping **chunks** (one vector each)
- Chunk size & overlap affect retrieval quality (tune later)
- This is the **ingestion** phase, step 1

> ### 🎤 Instructor Script
>
> Every RAG system starts by turning documents into chunks. We load two security files — Omar's SSRF explainer and our cybersecurity knowledge base — and split each into overlapping chunks of a few hundred characters. Each chunk becomes one vector we can retrieve, and the overlap keeps ideas that straddle a boundary intact. The printed chunk count is how many searchable units we'll store. Chunk size and overlap are the first knobs you'll tune if answers come back vague.

In [2]:
# a small function that cuts a long text into overlapping chunks
def split_into_chunks(text, chunk_size=600, overlap=100):
    pieces = []
    start = 0
    while start < len(text):
        piece = text[start : start + chunk_size].strip()
        if piece != "":
            pieces.append(piece)
        start = start + chunk_size - overlap
    return pieces

files = ["ssrf.txt", "cybersecurity_kb.md"]

chunks = []          # the text of every chunk
sources = []         # which file each chunk came from
for filename in files:
    if not os.path.exists(filename):
        print("skipping (not found):", filename)
        continue
    text = open(filename, encoding="utf-8").read()
    file_chunks = split_into_chunks(text)
    for piece in file_chunks:
        chunks.append(piece)
        sources.append(filename)
    print(filename, "->", len(file_chunks), "chunks")

print("\nTotal chunks to store:", len(chunks))

ssrf.txt -> 11 chunks
cybersecurity_kb.md -> 11 chunks

Total chunks to store: 22


# 🗄️ Step 2 — Embed & Store in ChromaDB

- Embed every chunk with our local model
- Store vectors + text + **metadata** (source file) in Chroma
- Use the **cosine** space so distance = semantic closeness
- Now we have a searchable vector knowledge base

> ### 🎤 Instructor Script
>
> Next we build the knowledge base. We embed all the chunks with our local model and load them into a Chroma collection, storing each chunk's text and its source file as metadata. We tell the collection to use cosine distance, so a smaller distance means closer in meaning. After this one-time ingestion the collection is a searchable vector database — exactly what Omar's basic_rag_part1 builds, minus the LangChain wrapper so you can see every piece.

In [3]:
# create the vector database (in memory, using cosine distance)
client = chromadb.EphemeralClient()
collection = client.get_or_create_collection("security_kb", metadata={"hnsw:space": "cosine"})

# build an id and a source-label for each chunk
ids = []
labels = []
for i in range(len(chunks)):
    ids.append("c" + str(i))
    labels.append({"source": sources[i]})

# add the chunks (text + embeddings + labels) to the database
collection.add(ids=ids, documents=chunks, embeddings=embed(chunks), metadatas=labels)
print("Vector store ready. Stored chunks:", collection.count())

Vector store ready. Stored chunks: 22


# 🔍 Step 3 — Retrieve Relevant Context

- Embed the user's question with the **same** model
- Ask Chroma for the top-k closest chunks
- Apply a **distance threshold** to drop weak matches
- This is the **query** phase — retrieval by meaning

> ### 🎤 Instructor Script
>
> Now the query phase. We embed the question with the same model, ask Chroma for the closest few chunks, and keep only those under a distance threshold so we don't feed the model junk when nothing is truly relevant. We reuse Omar's SSRF question. Watch the retrieved chunks and their distances — the smallest distances are the passages that actually answer it. Tuning k and the threshold is where much of RAG's quality comes from.

In [4]:
# find the chunks closest in meaning to a question
def retrieve(query, how_many=4, max_distance=0.75):
    results = collection.query(query_embeddings=embed([query]), n_results=how_many)
    found_docs = results["documents"][0]
    found_dists = results["distances"][0]
    found_meta = results["metadatas"][0]

    good_hits = []
    for k in range(len(found_docs)):
        if found_dists[k] <= max_distance:          # keep only close-enough matches
            good_hits.append((found_docs[k], found_dists[k], found_meta[k]))
    return good_hits

query = "What is SSRF? Provide an example of an SSRF attack."
print("Query:", query, "\n\nRetrieved context:")
for doc, dist, meta in retrieve(query):
    print("  (distance", round(dist, 3), "from", meta["source"], ")", doc[:90].strip(), "...")

Query: What is SSRF? Provide an example of an SSRF attack. 

Retrieved context:
  (distance 0.307 from ssrf.txt ) ed to perform an SSRF attack. Take a look at the XXE cheat sheet to learn how to prevent e ...
  (distance 0.366 from ssrf.txt ) ed, and if poorly handled, can perform specific injection attacks.
Overview of a SSRF Comm ...
  (distance 0.386 from ssrf.txt ) Server-Side Request Forgery Prevention Cheat Sheet
Introduction
The objective of the cheat ...
  (distance 0.389 from ssrf.txt ) teract with the internal/external network or the machine itself. One of the enablers for t ...


# 🔗 Step 4 — Augment the Prompt

- Combine retrieved context + the question into one prompt
- Instruct the model to answer **only from the context**
- Tag answers `[RAG]` (from docs) or `[LLM]` (no context found)
- This grounds the answer and makes hallucination visible

> ### 🎤 Instructor Script
>
> Augmentation is just prompt assembly. We paste the retrieved chunks into a system instruction that says answer only from this context, and admit it if the answer isn't there. We borrow Omar's habit of tagging the reply — RAG when it used the documents, LLM when it fell back — so grounding is visible at a glance. We print the fully assembled prompt: no magic, just a well-built string the model will receive.

In [5]:
SYSTEM = (
    "You are a cybersecurity assistant. Answer the question using ONLY the context below.\n"
    "If the answer is in the context, prefix your reply with [RAG]. If the context is empty or does "
    "not contain the answer, say so and prefix with [LLM].\n\n# Context\n{context}"
)

def build_prompt(query, context):
    system_message = SYSTEM.format(context=context or "(no relevant context found)")
    return system_message, query

# join the retrieved chunks into one block of context text
hits = retrieve(query)
context_pieces = []
for doc, dist, meta in hits:
    context_pieces.append(doc)
context = "\n\n".join(context_pieces)

system, user = build_prompt(query, context)
print("=== SYSTEM PROMPT (assembled) ===\n", system[:600], "...\n")
print("=== USER ===\n", user)

=== SYSTEM PROMPT (assembled) ===
 You are a cybersecurity assistant. Answer the question using ONLY the context below.
If the answer is in the context, prefix your reply with [RAG]. If the context is empty or does not contain the answer, say so and prefix with [LLM].

# Context
ed to perform an SSRF attack. Take a look at the XXE cheat sheet to learn how to prevent exposure to XXE.
Cases
Depending on the application’s functionality and requirements, there are two basic cases in which SSRF can happen:

Application can send requests only to identified and trusted applications: This occurs when an allowlist approach is available. ...

=== USER ===
 What is SSRF? Provide an example of an SSRF attack.


# 💬 Step 5 — Generate a Grounded Answer

- Send the augmented prompt to an LLM
- OpenAI if a key is set; otherwise print the prompt (offline)
- You could also point this at your **Module 3 local model**
- The answer is grounded in retrieved security docs

> ### 🎤 Instructor Script
>
> The last step is generation: send the augmented prompt to a model and get an answer grounded in the retrieved documents. If an OpenAI key is set, the cell calls it; if not, it prints the prompt so the pipeline still completes offline — and you could point this at the local model you hosted in Module 3, the free and private option. Either way, the model now answers from your SSRF and knowledge-base documents, not from frozen training memory.

In [6]:
def generate(system, user):
    if os.getenv("OPENAI_API_KEY"):
        from openai import OpenAI
        r = OpenAI().chat.completions.create(
            model="gpt-4o-mini", temperature=0,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
        return r.choices[0].message.content
    return ("[offline] No OPENAI_API_KEY set. Would send the assembled prompt above to an LLM "
            "(OpenAI, or your Module 3 local model).")

print(generate(system, user))

[offline] No OPENAI_API_KEY set. Would send the assembled prompt above to an LLM (OpenAI, or your Module 3 local model).


# 🤖 Step 6 — The Complete RAG Function

- Wrap retrieve → augment → generate into one `ask()`
- In-domain question → grounded `[RAG]` answer
- Out-of-domain question → no context → honest `[LLM]` fallback
- That contrast is the whole point of RAG

> ### 🎤 Instructor Script
>
> Finally we wrap the three steps into a single ask function — the complete basic RAG application. We run it on an in-domain question, where it pulls real context and answers from the documents, and an out-of-domain question, where retrieval finds nothing and the system falls back honestly instead of inventing an answer. That contrast — grounded when your data is relevant, graceful when it isn't — is exactly why RAG matters in security, where a confident wrong answer is dangerous.

In [7]:
# the complete RAG function: retrieve -> augment -> generate
def ask(query):
    hits = retrieve(query)

    # join the retrieved chunks into one context block
    context_pieces = []
    for doc, dist, meta in hits:
        context_pieces.append(doc)
    context = "\n\n".join(context_pieces)

    system, user = build_prompt(query, context)
    print("Q:", query)
    if len(hits) > 0:
        print("  grounded? YES (found relevant context)")
    else:
        print("  grounded? NO (no relevant context)")
    print("  answer:", generate(system, user)[:300], "\n")

ask("What is SSRF and how can it be exploited?")     # in the documents  -> [RAG]
ask("What is the capital of France?")                # not in the documents -> [LLM] fallback

Q: What is SSRF and how can it be exploited?
  grounded? YES (found relevant context)
  answer: [offline] No OPENAI_API_KEY set. Would send the assembled prompt above to an LLM (OpenAI, or your Module 3 local model). 

Q: What is the capital of France?
  grounded? NO (no relevant context)
  answer: [offline] No OPENAI_API_KEY set. Would send the assembled prompt above to an LLM (OpenAI, or your Module 3 local model). 



# ✅ Summary — The RAG Recipe

- Load → **chunk** documents (ingestion)
- **Embed** chunks → store in a **vector DB** (ChromaDB)
- **Retrieve** top-k by meaning (with a threshold)
- **Augment** a prompt with the context
- **Generate** a grounded, tagged answer

> ### 🎤 Instructor Script
>
> You built a complete RAG system from scratch: load and chunk the documents, embed them into a Chroma vector store, retrieve the most relevant chunks, augment a prompt with that context, and generate a grounded answer with an honest fallback. This is the same pipeline Omar teaches, and the same skeleton scales from this in-memory demo to a production store with millions of documents. Next module, you'll rebuild it with LangChain and see how a framework streamlines the wiring you just did by hand.

In [8]:
print("RAG pipeline: documents -> chunk -> embed -> ChromaDB -> retrieve -> augment -> generate")
print("You built every step without a framework. Next module: the LangChain version.")

RAG pipeline: documents -> chunk -> embed -> ChromaDB -> retrieve -> augment -> generate
You built every step without a framework. Next module: the LangChain version.
